In [4]:
import json
from pathlib import Path
from collections import Counter

In [14]:
# File/Folder Paths
# PROJECT_ROOT = Path(__file__).resolve().parents[1]
PROJECT_ROOT = Path.cwd().parents[0]
ANNOTATION_FILE = (PROJECT_ROOT/"data"/"SSDD"/"annotations"/"train.json")

In [16]:
# Loading the annotations
with open(ANNOTATION_FILE, "r") as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]
categories = data["categories"]

print(f"Number of Images: {len(images)}")
print(f"Number of Annotations: {len(annotations)}")
print(f"Number of Categories: {len(categories)}")

Number of Images: 928
Number of Annotations: 2041
Number of Categories: 1


In [17]:
# Image dimensions
widths = [img["width"] for img in images]
heights = [img["height"] for img in images]

print("\nImage dimensions:")

print(f"Width  - min : {min(widths)}")
print(f"Width  - max : {max(widths)}")

print(f"Height - min : {min(heights)}")
print(f"Height - max : {max(heights)}")


Image dimensions:
Width  - min : 214
Width  - max : 653
Height - min : 190
Height - max : 526


In [19]:
# Counting ships per image
ships_per_image = Counter(
    annotation["image_id"] for annotation in annotations
)

ship_counts = list(ships_per_image.values())

print(f"Minimum ships/image: {min(ship_counts)}")
print(f"Maximum ships/image: {max(ship_counts)}")
print(f"Average ships/image: {sum(ship_counts)/len(ship_counts):.2f}")

Minimum ships/image: 1
Maximum ships/image: 29
Average ships/image: 2.20


In [23]:
# Images without ships
img_ids = {img["id"] for img in images}
annotated_img_ids = set(ships_per_image.keys())

images_without_ship = img_ids - annotated_img_ids

print(f"Images without Annotations: {len(images_without_ship)}")

Images without Annotations: 0


In [24]:
# Bounding Box Sizes
bbox_widths = []
bbox_heights = []
bbox_areas = []

for annotation in annotations:
    _, _, width, height = annotation["bbox"]

    bbox_widths.append(width)
    bbox_heights.append(height)
    bbox_areas.append(annotation["area"])

print("\nBounding boxes:")

print(f"Average bbox width  : {sum(bbox_widths) / len(bbox_widths):.2f}")
print(f"Average bbox height : {sum(bbox_heights) / len(bbox_heights):.2f}")
print(f"Average bbox area   : {sum(bbox_areas) / len(bbox_areas):.2f}")


print("\nCategories:")

for category in categories:
    print(
        f"ID: {category['id']} | "
        f"Name: {category['name']}"
    )


Bounding boxes:
Average bbox width  : 33.67
Average bbox height : 38.53
Average bbox area   : 849.46

Categories:
ID: 0 | Name: ship


In [26]:
# Trying to identify ships' sizes
import json
import numpy as np

with open(ANNOTATION_FILE, "r") as f:
    data = json.load(f)

bbox_widths = []
bbox_heights = []
bbox_areas = []

for ann in data["annotations"]:
    x, y, w, h = ann["bbox"]

    bbox_widths.append(w)
    bbox_heights.append(h)
    bbox_areas.append(w * h)

for name, values in [
    ("Width", bbox_widths),
    ("Height", bbox_heights),
    ("Area", bbox_areas),
]:
    values = np.array(values)

    print(f"\n{name}:")
    print(f"Min    : {values.min():.2f}")
    print(f"25%    : {np.percentile(values, 25):.2f}")
    print(f"Median : {np.median(values):.2f}")
    print(f"75%    : {np.percentile(values, 75):.2f}")
    print(f"Max    : {values.max():.2f}")
    print(f"Mean   : {values.mean():.2f}")


Width:
Min    : 4.21
25%    : 13.02
Median : 21.70
75%    : 40.10
Max    : 380.00
Mean   : 33.67

Height:
Min    : 4.70
25%    : 16.67
Median : 28.68
75%    : 49.30
Max    : 307.12
Mean   : 38.53

Area:
Min    : 26.01
25%    : 286.04
Median : 597.33
75%    : 1618.97
Max    : 55348.15
Mean   : 1643.83


In [27]:
ships_per_image = {}

for ann in data["annotations"]:
    image_id = ann["image_id"]

    ships_per_image[image_id] = (
        ships_per_image.get(image_id, 0) + 1
    )

counts = np.array(list(ships_per_image.values()))

print("\nShips per annotated image:")
print("Min   :", counts.min())
print("Median:", np.median(counts))
print("Mean  :", counts.mean())
print("Max   :", counts.max())


Ships per annotated image:
Min   : 1
Median: 1.0
Mean  : 2.199353448275862
Max   : 29
